In [4]:
import pandas as pd


In [5]:
df = pd.read_excel("/content/base_original_pmdf.xlsx")

In [3]:
df.head(5)

,Unnamed: 0,ANO: 2024,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
0,NaN,INCIDENCIA: LEI 11.340 - MARIA DA PENHA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NUMERO,DATA HORA,INCIDENCIA,CIDADE,BAIRRO,QUADRA,COMPLEMENTO,HISTORICO,LATITUDE,LONGITUDE
3,NaN,2024010100698028682,2024-01-01 00:05:00,LEI 11.340 (MARIA DA PENHA),NUCLEO BANDEIRANTE,SEM_SETOR,2ª AVENIDA,APARTAMENTO 102,Foi passado uma situação de monitorado por tor...,"-15,86942","-47,96948"
4,NaN,2024010100691028683,2024-01-01 01:07:00,LEI 11.340 (MARIA DA PENHA),SAO SEBASTIAO,BAIRRO CENTRO,RUA 64A,CASA 31,Acionado para atendimento de ocorrência de cri...,"-15,94647","-48,06821"


In [6]:
df.columns = df.iloc[2]
df = df.iloc[3:].reset_index(drop=True)
df = df.iloc[:, 1:]

In [7]:
df.head(35)

2,NUMERO,DATA HORA,INCIDENCIA,CIDADE,BAIRRO,QUADRA,COMPLEMENTO,HISTORICO,LATITUDE,LONGITUDE
0,2024010100698028682,2024-01-01 00:05:00,LEI 11.340 (MARIA DA PENHA),NUCLEO BANDEIRANTE,SEM_SETOR,2ª AVENIDA,APARTAMENTO 102,Foi passado uma situação de monitorado por tor...,"-15,86942","-47,96948"
1,2024010100691028683,2024-01-01 01:07:00,LEI 11.340 (MARIA DA PENHA),SAO SEBASTIAO,BAIRRO CENTRO,RUA 64A,CASA 31,Acionado para atendimento de ocorrência de cri...,"-15,94647","-48,06821"
2,2024010100024028697,2024-01-01 00:58:00,LEI 11.340 (MARIA DA PENHA),SAMAMBAIA,SEM_SETOR,QR 403,CASA 20,"No dia 01 de janeiro de 2024, às 00:58, a viat...","-15,87031","-48,09789"
3,2024010100005028734,2024-01-01 02:53:00,LEI 11.340 (MARIA DA PENHA),TAGUATINGA,SETOR D SUL,QSD 33,LOTE 04 APTO 202,"Não há apto no lote 04, ou seja, endereço inco...","-15,94647","-48,06821"
4,2024010100691028741,2024-01-01 02:08:00,LEI 11.340 (MARIA DA PENHA),SAO SEBASTIAO,SEM_SETOR,NaN,VILA DO BOA,Acionado para atendimento de ocorrência com in...,"-15,94647","-48,06821"
5,2024010100005028747,2024-01-01 02:04:00,LEI 11.340 (MARIA DA PENHA),TAGUATINGA,SETOR F SUL,QSF 2,CASA 303,A Sra Daniela visivelmente aparentava ter prob...,"-15,94647","-48,06821"
6,2024010100005028748,2024-01-01 02:53:00,LEI 11.340 (MARIA DA PENHA),TAGUATINGA,SETOR E SUL,QSE 22,FAVELA AO LADO DA LINHA DO METRÔ,A equipe fez abordagem a dois homens suspeitos...,"-15,94647","-48,06821"
7,2024010100700028750,2024-01-01 01:57:00,LEI 11.340 (MARIA DA PENHA),RECANTO DAS EMAS,SEM_SETOR,QUADRA 511,CASA 23,"Quando em patrulhamento pelo Recanto das Emas,...","-15,93132","-48,10732"
8,2024010100691028788,2024-01-01 01:34:00,LEI 11.340 (MARIA DA PENHA),SAO SEBASTIAO,BAIRRO TRADICIONAL,NaN,"RUA 21, 41",Fomos acionados via COPOM para averiguar situa...,"-15,94647","-48,06821"
9,2024010100034028800,2024-01-01 08:40:00,LEI 11.340 (MARIA DA PENHA),PLANALTINA,SH ARAPOANGA_CONDOMINIO ARAPOANGA,QUADRA 15/17M,CASA 86 A,Quando em patrulhamento este prefixo foi acion...,"-15,6448","-47,63412"


In [8]:
import re

PALAVRAS_COMUNS = {
    "A", "AO", "AOS", "AS", "ATÉ", "ATE", "COM", "COMO", "DA", "DAS", "DE",
    "DO", "DOS", "E", "EM", "FOI", "PARA", "POR", "QUE", "SE", "SUA", "SUAS",
    "SEU", "SEUS", "UM", "UMA", "UNS", "UMAS", "NA", "NAS", "NO", "NOS",
    "NÃO", "NAO", "MAS", "OU", "ELE", "ELA", "ELES", "ELAS", "EU", "NÓS",
    "NOS", "ISSO", "ESTE", "ESTA", "ESSE", "ESSA", "AQUELE", "AQUELA",
    "VITIMA", "VÍTIMA", "AUTOR", "AUTORA", "EQUIPE", "LOCAL", "GUARNIÇÃO",
    "GUARNICAO", "HOSPITAL", "DELEGACIA", "POLICIA", "POLÍCIA", "AGRESSOR",
    "AGRESSÃO", "AGRESSAO", "OCORRÊNCIA", "OCORRENCIA", "PREFIXO", "AREA",
    "ÁREA", "APOIO", "CASA", "RUA", "ROSTO", "CABEÇA", "CABECA", "LADO",
    "MOMENTO", "OBJETIVO", "AÇÃO", "ACAO", "CHEGADA", "FATOS", "FATO",
    "FILHA", "FILHO", "FILHOS", "FILHAS", "PAI", "MÃE", "MAE", "PAIS",
    "NETOS", "AVÔ", "AVO", "VIZINHOS", "VIZINHO", "CRIANÇAS", "CRIANCAS",
    "CRIANÇA", "CRIANCA", "SOCORRO", "OBJETO", "TIJOLO", "ESTACA", "MURRO",
    "GOLPE", "IMPACTO", "CORTE", "SANGUE", "CHOQUE", "NERVOSA", "NERVOSO",
    "PRIMEIRO", "PRIMEIROS", "ATENDIMENTO", "MEDICO", "MÉDICO", "SOCORROS",
    "POSTERIORMENTE", "ANTERIORMENTE", "IMEDIATO", "IMEDIATA", "INSTANTES",
    "APOS", "APÓS", "ANTES", "DURANTE", "ENQUANTO", "SEGUIDA", "CONTINUO",
    "CONTÍNUO", "IMPORTANTE", "DESTACAR", "TENTATIVA", "TENTOU", "LOGROU",
    "EXITO", "ÊXITO", "SUCESSO", "PROVIDÊNCIAS", "PROVIDENCIAS", "MEDIDAS",
    "PROTETIVAS", "JUDICIARIO", "JUDICIÁRIO", "POLICIAL", "POLICIAIS",
    "PODER", "CURATELA", "CUIDADO", "RESPONSABILIZOU", "ZELO", "REGISTRO",
    "REGISTROU", "APURAÇÃO", "APURACAO", "LOCALIZADO", "LOCALIZAÇÃO",
    "LOCALIZACAO", "IDENTIFICAR", "QUALIFICAÇÃO", "QUALIFICACAO", "FOTO",
    "PATRULHAMENTO", "DILIGENCIAS", "DILIGÊNCIAS", "PROXIMIDADES",
    "INFRAESTRUTURA", "URBANISTICA", "PRECÁRIA", "PRECARIA", "BARRACOS",
    "CONJUNTO", "QUADRA", "CHACARA", "ENDEREÇO", "ENDERECO", "INFORMADO",
    "INFORMOU", "INFORMARAM", "NARRADOS", "IMINENTE", "PERIGO", "CIÊNCIA",
    "CIENCIA", "DESLOCOU", "DESLOCARAM", "ENCAMINHADA", "ENCAMINHOU",
    "ENCONTROU", "ENCONTRAR", "CONSTATAR", "CONSTATADO", "CONSEGUINDO",
    "RETORNAR", "MATAR", "EVADIU", "IMPEDIRAM", "INTERVIRAM", "GRITANDO",
    "SAIU", "PEGOU", "USOU", "CAIU", "DISCUTIU", "DESFERIU", "REALIZOU",
    "REPASSOU", "REQUEREU", "ENTREGOU", "COMPARECERAM", "SOLICITOU",
    "ACIONADA", "ACIONADOS", "AVERIGUAR", "POSSIVEL", "POSSÍVEL",
    "SITUAÇÃO", "SITUACAO", "VIOLÊNCIA", "VIOLENCIA", "DOMÉSTICA",
    "DOMESTICA", "AMEAÇA", "AMEACA", "CÁRCERE", "CARCERE", "PRIVADO",
    "PROFUNDO", "PROFUNDA", "PROLONGAVA", "PROVENIENTE", "APROXIMADAMENTE",
    "METADE", "QUANTIDADE", "GRANDE", "ESQUERDO", "DIREITO", "TESTA",
    "INCERTO", "CRIMINOSA", "PROSSEGUIMENTO", "EXTREMIDADE", "PONTIAGUDA",
    "ALCANCE", "APARENTAVA", "CEIFAR", "AGRESSÕES", "AGRESSOES", "ACERTAR",
    "TOMANDO", "RUMO", "PROMESSA", "CHAMADOS", "CBMDF", "COPOM", "CBM",
    "SGT", "SRA", "SR", "GAE", "AD", "UR", "GTOP", "PCDF", "FLAGRANTE",
    "AGENTE", "MULHER", "HOMEM", "PALAVRA", "CALAO", "CALHÃO", "CHUTE",
    "CIUMES", "CIÚMES", "PARTES", "PROVIDENCIAS", "CABIVEIS", "ESPOSA",
    "MARIDO", "SOGRA", "SOGRO", "GENRO", "NORA", "CUNHADA", "CUNHADO",
    "CONCUNHADA", "CONCUNHADO", "CRIANÇA", "CASAL", "CONDUZIDO", "PRESO",
    "ALGEMAS", "EMBRIAGUEZ", "EMBRIAGADO", "ALCOOL", "ÁLCOOL", "ARMA",
    "FOGO", "BUSCA", "DOMICILIAR", "PRISÃO", "PRISAO", "PROCEDIMENTO",
    "LAVRATURA", "DELEGACIA", "OCORRENCIA", "OCORRÊNCIA",
}

NOMES_BRASILEIROS = [
    # Femininos
    "Ana", "Maria", "Fernanda", "Juliana", "Amanda", "Camila", "Beatriz",
    "Larissa", "Leticia", "Letícia", "Patricia", "Patrícia", "Gabriela",
    "Rafaela", "Renata", "Vanessa", "Claudia", "Cláudia", "Sandra", "Simone",
    "Mariana", "Adriana", "Cristiane", "Cristina", "Luciana", "Rosana",
    "Rosangela", "Rosângela", "Fatima", "Fátima", "Regina", "Vera", "Lucia",
    "Lúcia", "Eliane", "Elisangela", "Elisângela", "Aparecida", "Aline",
    "Bianca", "Bruna", "Carolina", "Carla", "Daniela", "Deborah", "Debora",
    "Débora", "Edilene", "Edna", "Elza", "Erica", "Érica", "Flavia", "Flávia",
    "Gisele", "Giselle", "Giovana", "Gilvania", "Gilvânia", "Iara", "Ingrid",
    "Isabela", "Isabella", "Jessica", "Jéssica", "Josefa", "Joyce", "Julia",
    "Júlia", "Juliane", "Karina", "Katia", "Kátia", "Keyna", "Laura", "Leia",
    "Leidiana", "Leidiane", "Lilian", "Liliane", "Livia", "Lívia", "Lorena",
    "Luana", "Luiza", "Luíza", "Marta", "Melissa", "Michele", "Michelle",
    "Monica", "Mônica", "Naiara", "Natalia", "Natália", "Nathalia", "Nathália",
    "Nilma", "Paloma", "Paula", "Priscila", "Raquel", "Rebeca", "Rebecca",
    "Rita", "Roberta", "Rosa", "Sabrina", "Sara", "Sarah", "Silvia", "Sílvia",
    "Solange", "Sonia", "Sônia", "Sueli", "Suely", "Suzana", "Suzane",
    "Tais", "Taís", "Tatiana", "Tatiane", "Telma", "Tereza", "Teresa",
    "Thais", "Thaís", "Thalia", "Thália", "Valdirene", "Valeria", "Valéria",
    "Viviane", "Viviani", "Wanda", "Yasmin", "Yara", "Manoela", "Manuela",
    "Kelma", "Gleice", "Gleicy", "Jaqueline", "Jacqueline", "Elaine",
    "Valentina", "Helena", "Alice", "Elisa", "Clarice", "Aurora", "Maite",
    "Maitê", "Catarina", "Heloisa", "Heloísa", "Taina", "Tainá", "Sofia",
    "Daiane", "Monique", "Nicole", "Barbara", "Bárbara", "Keitiany",
    "Ludimila", "Karolayne", "Jarinete", "Andressa", "Rosemery", "Ariane",
    "Francisca", "Benedita", "Sebastiana", "Antonia", "Antônia", "Carmelita",
    "Inacia", "Inácia", "Dalva", "Nilza", "Vania", "Vânia", "Noemia",
    "Noêmia", "Odete", "Zilda", "Efigenia", "Efigênia", "Jandira", "Marlene",
    "Ivone", "Dirce", "Celia", "Célia", "Nair", "Neide", "Zuleide", "Zulmira",
    "Raimunda", "Quiteria", "Quitéria", "Gertrudes", "Edeltrudes", "Doralice",
    "Janete", "Arlete", "Jucimara", "Rosilene", "Cidalia", "Cidália",
    "Magnolia", "Magnólia", "Zoraide", "Iracema", "Lindalva", "Jucelia",
    "Jucélia", "Valdenia", "Valdênia", "Josenilda", "Rosicleide", "Claudete",
    "Marilene", "Jucilene", "Cleonice", "Dejanira", "Terezinha", "Luzinete",
    "Ivonete", "Valquiria", "Valquíria", "Nadia", "Nádia",
    # Masculinos
    "Adam", "Adilson", "Adriano", "Alexandre", "Alex", "Anderson", "Andre",
    "André", "Antonio", "Antônio", "Bruno", "Carlos", "Caio", "Claudio",
    "Cláudio", "Daniel", "Danilo", "David", "Diego", "Douglas", "Eduardo",
    "Edson", "Elias", "Emanuel", "Emerson", "Evandro", "Fabio", "Fábio",
    "Felipe", "Filipe", "Fernando", "Francisco", "Frederico", "Gabriel",
    "Giordano", "Gilberto", "Guilherme", "Gustavo", "Henrique", "Hugo",
    "Igor", "Ivan", "Iron", "Izaque", "Jefferson", "Jonathan", "Jorge",
    "Jose", "José", "Joao", "João", "Julio", "Júlio", "Junior", "Júnior",
    "Kaique", "Kaio", "Leonardo", "Leandro", "Lucas", "Luciano", "Luis",
    "Luís", "Luiz", "Marcelo", "Marcio", "Márcio", "Marcos", "Mateus",
    "Matheus", "Mauricio", "Maurício", "Miguel", "Murilo", "Nelson",
    "Nicolas", "Nicolau", "Paulo", "Pedro", "Rafael", "Raphael", "Ricardo",
    "Roberto", "Rodrigo", "Romulo", "Rômulo", "Rogerio", "Rogério", "Ruan",
    "Samuel", "Sergio", "Sérgio", "Silas", "Tiago", "Thiago", "Vagner",
    "Valter", "Victor", "Vinicius", "Vinícius", "Wagner", "Wallace",
    "Wallyson", "Wander", "Washington", "Wellington", "Wellinton", "Welder",
    "William", "Willian", "Kelmon", "Kelvin", "Kevin", "Renato", "Otavio",
    "Otávio", "Davi", "Enzo", "Theo", "Arthur", "Benjamin", "Noah",
    "Heitor", "Lorenzo", "Raul", "Tadeu", "Bento", "Cicero", "Cícero",
    "Baltazar", "Ezequiel", "Ismael", "Adalberto", "Osvaldo", "Severino",
    "Raimundo", "Valdemar", "Hermenegildo", "Aristides", "Leonidas",
    "Leônidas", "Eustaquio", "Eustáquio", "Genesio", "Genésio", "Onofre",
    "Epifanio", "Epifânio", "Deolindo", "Juvenal", "Agenor", "Moacir",
    "Ubirajara", "Jurandir", "Valdir", "Vanderlei", "Ademir", "Valmir",
    "Olegario", "Olegário", "Wenceslau", "Alcides", "Florisvaldo",
    "Gervasio", "Gervásio", "Lindomar", "Aureliano", "Bernardino",
    "Claudionor", "Djalma", "Edivaldo", "Josue", "Josué", "Josias",
    "Salomao", "Salomão", "Tobias", "Zacarias", "Emanoel", "Abdias",
    "Jeferson", "Edmilson", "Amarildo", "Josenildo", "Roberval", "Gleidson",
    "Uelinton", "Halrisson", "Atylla", "Reginaldo", "Zilli", "Souto",
    # Sobrenomes comuns
    "Silva", "Santos", "Oliveira", "Souza", "Sousa", "Rodrigues", "Alves",
    "Ferreira", "Pereira", "Lima", "Gomes", "Costa", "Ribeiro", "Martins",
    "Carvalho", "Almeida", "Lopes", "Rocha", "Nascimento", "Moreira",
    "Nunes", "Barbosa", "Cavalcanti", "Dias", "Cardoso", "Moura", "Ramos",
    "Campos", "Pinto", "Freitas", "Teixeira", "Monteiro", "Vieira",
    "Fonseca", "Azevedo", "Mendes", "Cunha", "Pinheiro", "Medeiros",
    "Batista", "Araujo", "Araújo", "Cruz", "Melo", "Brito", "Xavier",
    "Simoes", "Simões", "Iacozzilli", "Caseiro", "Giordano",
]

_vistos = set()
_NOMES_UNICOS = []
for n in NOMES_BRASILEIROS:
    if n.lower() not in _vistos:
        _vistos.add(n.lower())
        _NOMES_UNICOS.append(n)

_NOMES_COMPILED = [
    re.compile(r'\b' + re.escape(n) + r'\b', re.IGNORECASE)
    for n in _NOMES_UNICOS
]

WHITELIST = [
    r'Lei\s+Maria\s+da\s+Penha', r'Maria\s+da\s+Penha',
    r'Código\s+Penal', r'Ministério\s+Público',
    r'Boletim\s+de\s+Ocorrência', r'Polícia\s+Civil', r'Polícia\s+Militar',
    r'Corpo\s+de\s+Bombeiros', r'Defensoria\s+Pública', r'Vara\s+Criminal',
    r'Hospital\s+Regional', r'COPOM', r'CBM/DF', r'CBM', r'CBMDF',
    r'Plano\s+Piloto', r'Asa\s+Sul', r'Asa\s+Norte',
    r'Brasília', r'Gama', r'Taguatinga', r'Brazlândia',
    r'Sobradinho\s+II', r'Sobradinho', r'Planaltina', r'Paranoá',
    r'Núcleo\s+Bandeirante', r'Ceilândia', r'Guará', r'Cruzeiro',
    r'Samambaia\s+Norte', r'Samambaia\s+Sul', r'Samambaia',
    r'Santa\s+Maria', r'São\s+Sebastião', r'Recanto\s+das\s+Emas',
    r'Lago\s+Sul', r'Riacho\s+Fundo\s+II', r'Riacho\s+Fundo',
    r'Lago\s+Norte', r'Candangolândia', r'Águas\s+Claras',
    r'Sudoeste', r'Octogonal', r'Varjão', r'Park\s+Way',
    r'Estrutural', r'Jardim\s+Botânico', r'Itapoã',
    r'Vicente\s+Pires', r'Fercal', r'Noroeste', r'Distrito\s+Federal',
    r'Delegacia\s+de\s+Pol[íi]cia',
    r'1[ºª]\s+Delegacia', r'26[ºª]\s+Delegacia', r'30[ºª]\s+Delegacia',
]

_WHITELIST_COMPILED = [
    (re.compile(p, re.IGNORECASE), f"__WL{i}__")
    for i, p in enumerate(WHITELIST)
]

CONTEXTO = re.compile(
    r'\b(solicitante|v[íi]tima|autor[a]?|acusad[oa]|suspeito[a]?|'
    r'testemunha|comunicante|requerente|agente|policial|padrasto|madrasta|'
    r'companheiro|companheira|namorado[a]?|irm[ãa]o?|filh[oa]|'
    r'esposo|esposa|marido|sogr[ao]|genro|nora|cunhad[ao]|concunhad[ao]|'
    r'm[ãa]e\s+d[ao]|pai\s+d[ao]|'
    r'posteriormente\s+identificad[oa]\s+como(?:\s+sendo)?|'
    r'identificad[oa]\s+como(?:\s+sendo)?|'
    r'pel[oa]\s+agente|'
    r'senhora|senhor|srta\.?|sra\.?|sr\.?|sr[aª]\.?|sr[oº]\.?|sgt\.?|cb\.?|sd\.?|'
    r'exma?\.?\s*sra?\.?|dr\.?|dra\.?|'
    r'com\s+[oa]|juntamente\s+com|contato\s+com)'
    r'[\s,\(]+'
    r'([A-ZÁÉÍÓÚÂÊÔÃÕÀÜÇ][A-ZÁÉÍÓÚÂÊÔÃÕÀÜÇa-záéíóúâêôãõàüç]+'
    r'(?:\s+(?:da?|de|do|dos|das)\s+[A-ZÁÉÍÓÚÂÊÔÃÕÀÜÇ][a-záéíóúâêôãõàüç]+|'
    r'\s+[A-ZÁÉÍÓÚÂÊÔÃÕÀÜÇ][A-ZÁÉÍÓÚÂÊÔÃÕÀÜÇa-záéíóúâêôãõàüç]+)*)',
    re.IGNORECASE
)

_CPF      = re.compile(r'\b\d{3}[\.\s]?\d{3}[\.\s]?\d{3}[\-\s]?\d{2}\b')
_CNPJ     = re.compile(r'\b\d{2}[\.\s]?\d{3}[\.\s]?\d{3}[\/\.]?\d{4}[\-]?\d{2}\b')
_EMAIL    = re.compile(r'[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}')
_TELEFONE = re.compile(r'\(?\d{2}\)?\s?\d{4,5}[\-\s]?\d{4}')
_CODIGO   = re.compile(r'\b\d{5,}\b')
_MATRICULA = re.compile(
    r'\b(?:mat(?:r[íi]cula)?\.?\s*:?\s*)[\d\w\-\.\/]{4,20}\b'
    r'|\b\d{1,3}(?:\.\d{3})+[\-\/]\d{1,2}\b',
    re.IGNORECASE
)
_NOME_COMPOSTO = re.compile(
    r'\b[A-ZÁÉÍÓÚÂÊÔÃÕÀÜÇ][a-záéíóúâêôãõàüç]+'
    r'(?:\s+(?:da?|de|do|dos|das|e))*'
    r'(?:\s+[A-Za-záéíóúâêôãõàüçÁÉÍÓÚÂÊÔÃÕÀÜÇ][a-záéíóúâêôãõàüç]+){1,4}\b'
)
_NOME_POS_PONTUACAO = re.compile(
    r'(?<=[\.!\?])\s*([A-ZÁÉÍÓÚÂÊÔÃÕÀÜÇ][a-záéíóúâêôãõàüç]{2,})\b'
)
_CAIXA_ALTA_CONTEXTO = re.compile(
    r'\b(SRA?\.?|SRTA\.?|SGT\.?|SENHORA|SENHOR|AGENTE|VITIMA|VÍTIMA|'
    r'SOLICITANTE|AUTOR[A]?|SUSPEITO[A]?|COMPANHEIRO|COMPANHEIRA|'
    r'NAMORADO[A]?|IRM[AÃ]O?|FILHO[A]?|PADRASTO|MAE|MÃE|PAI|AVÔ|AVO|'
    r'ESPOSA|ESPOSO|MARIDO|SOGR[AO]|GENRO|CUNHAD[AO]|CONCUNHAD[AO]|'
    r'MAE\s+D[AO]|PAI\s+D[AO]|'
    r'PELO\s+AGENTE|PELA\s+AGENTE|'
    r'POSTERIORMENTE\s+IDENTIFICAD[OA]\s+COMO(?:\s+SENDO)?|'
    r'IDENTIFICAD[OA]\s+COMO(?:\s+SENDO)?)'
    r'[\s,\(]+'
    r'([A-ZÁÉÍÓÚÂÊÔÃÕÀÜÇ][A-ZÁÉÍÓÚÂÊÔÃÕÀÜÇa-záéíóúâêôãõàüç]+'
    r'(?:\s+(?:DA?|DE|DO|DOS|DAS)\s+[A-ZÁÉÍÓÚÂÊÔÃÕÀÜÇ][A-ZÁÉÍÓÚÂÊÔÃÕÀÜÇ]+|'
    r'\s+[A-ZÁÉÍÓÚÂÊÔÃÕÀÜÇ][A-ZÁÉÍÓÚÂÊÔÃÕÀÜÇ]+)*)'
)


def _substituir_nomes_caixa_alta(texto):
    def _sub(match):
        palavra = match.group(2)
        if palavra.upper() in PALAVRAS_COMUNS:
            return match.group(0)
        return match.group(1) + ' [NOME]'
    return _CAIXA_ALTA_CONTEXTO.sub(_sub, texto)


def _texto_e_caixa_alta(texto):
    letras = [c for c in texto if c.isalpha()]
    if not letras:
        return False
    return sum(1 for c in letras if c.isupper()) / len(letras) > 0.70


def anonimizar_coluna(df, coluna="HISTORICO"):

    def proteger_whitelist(texto):
        protegidos = {}
        for regex, placeholder in _WHITELIST_COMPILED:
            encontrado = regex.search(texto)
            if encontrado:
                protegidos[placeholder] = encontrado.group(0)
                texto = regex.sub(placeholder, texto)
        return texto, protegidos

    def restaurar_whitelist(texto, protegidos):
        for placeholder, original in protegidos.items():
            texto = texto.replace(placeholder, original)
        return texto

    def anonimizar_texto(texto):
        if not isinstance(texto, str):
            return texto

        caixa_alta = _texto_e_caixa_alta(texto)
        texto, protegidos = proteger_whitelist(texto)

        texto = _CPF.sub('[CPF]', texto)
        texto = _CNPJ.sub('[CNPJ]', texto)
        texto = _MATRICULA.sub('[CODIGO]', texto)
        texto = _EMAIL.sub('[EMAIL]', texto)
        texto = _TELEFONE.sub('[TELEFONE]', texto)
        texto = _CODIGO.sub('[CODIGO]', texto)

        # Lista de nomes e sobrenomes conhecidos
        for regex_nome in _NOMES_COMPILED:
            texto = regex_nome.sub('[NOME]', texto)

        if caixa_alta:
            texto = _substituir_nomes_caixa_alta(texto)
        else:
            texto = _NOME_COMPOSTO.sub('[NOME]', texto)
            texto = CONTEXTO.sub(lambda m: f"{m.group(1)} [NOME]", texto)
            texto = _NOME_POS_PONTUACAO.sub('[NOME]', texto)

        texto = restaurar_whitelist(texto, protegidos)
        return texto

    df[coluna] = df[coluna].apply(anonimizar_texto)
    return df